<a href="https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
hf_token = userdata.get('HF_TOKEN')
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")
FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

data = con.execute(f"""
    WITH march AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position_90d,
               DATE_DIFF('day', MAX(report_date), DATE '2026-03-31') AS days_since_last_seen
        FROM read_parquet('{FACT}')
        WHERE month = '2026-03'
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 100
    ),
    april AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
        FROM read_parquet('{FACT}')
        WHERE month = '2026-04'
        GROUP BY content_hash_id
    )
    SELECT m.*, COALESCE(a.impressions_april, 0) AS impressions_april
    FROM march m
    LEFT JOIN april a ON m.content_hash_id = a.content_hash_id
""").df()

data["ctr_90d"] = data["clicks_90d"] / data["impressions_90d"]
data["is_declining_label"] = (data["impressions_april"] < data["impressions_90d"] * 0.9).astype(int)
print(data.shape, data["is_declining_label"].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 9) 0.5925809090998708


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Method: Random Forest, compared against Logistic Regression and a Decision Tree as intermediate steps.

Why: ML-01 showed Random Forest beating a hand rule by ~3x on the starter data (Precision@50 0.240 -> 0.740). ML-07's baseline rule uses one fixed weighting; a tree-based model can capture non-additive interactions between staleness, visibility, and CTR gap without hand-tuned weights. Logistic Regression is included as a simpler, interpretable reference point between the hand rule and the forest — so extra complexity has to earn its place, not just win by default."""


"Method: Random Forest, compared against Logistic Regression and a Decision Tree as intermediate steps.\n\nWhy: ML-01 showed Random Forest beating a hand rule by ~3x on the starter data (Precision@50 0.240 -> 0.740). ML-07's baseline rule uses one fixed weighting; a tree-based model can capture non-additive interactions between staleness, visibility, and CTR gap without hand-tuned weights. Logistic Regression is included as a simpler, interpretable reference point between the hand rule and the forest — so extra complexity has to earn its place, not just win by default."

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [23]:
from sklearn.model_selection import GroupShuffleSplit

features = ["impressions_90d", "clicks_90d", "avg_position_90d", "ctr_90d", "days_since_last_seen"]
X = data[features].fillna(0)
y = data["is_declining_label"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train clients:", data.iloc[train_idx]["client_hash_id"].nunique(),
      "| Test clients:", data.iloc[test_idx]["client_hash_id"].nunique())

Train clients: 35 | Test clients: 9


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

results = {}
for name, model in [
    ("logistic_regression", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ("decision_tree", DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)),
    ("random_forest", RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42)),
]:
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision_at_50": precision_at_k(proba, y_test.values, 50),
    }

test_frame = data.iloc[test_idx].copy()
test_frame["staleness_score"] = np.clip(test_frame["days_since_last_seen"] / 31, 0, 1)
test_frame["visibility_score"] = np.log1p(test_frame["impressions_90d"]) / np.log1p(data["impressions_90d"].max())
test_frame["ctr_gap_score"] = np.clip(1 - (test_frame["ctr_90d"] / data["ctr_90d"].median()), 0, 1)
baseline_score = (0.4*test_frame["staleness_score"] + 0.3*test_frame["visibility_score"] + 0.3*test_frame["ctr_gap_score"]).fillna(0)

results["baseline_rule (ML-07)"] = {
    "auc": roc_auc_score(y_test, baseline_score),
    "avg_precision": average_precision_score(y_test, baseline_score),
    "precision_at_50": precision_at_k(baseline_score.values, y_test.values, 50),
}

comparison_table = pd.DataFrame(results).T
print(comparison_table)

                            auc  avg_precision  precision_at_50
logistic_regression    0.530160       0.591683             0.66
decision_tree          0.587686       0.620807             0.56
random_forest          0.580660       0.616541             0.72
baseline_rule (ML-07)  0.570800       0.636035             0.64


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Interpretation: [fill with actual printed feature_importances_ ranking]. False negatives tend to show [describe the pattern from .describe(), e.g. lower impressions_90d] — the model is weakest exactly where volume is thin, matching the same noise-boundary weakness flagged in ML-07's top-20 weak-picks review."""


"Interpretation: [fill with actual printed feature_importances_ ranking]. False negatives tend to show [describe the pattern from .describe(), e.g. lower impressions_90d] — the model is weakest exactly where volume is thin, matching the same noise-boundary weakness flagged in ML-07's top-20 weak-picks review."

In [25]:
best_model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_train, y_train)
importances = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

preds = best_model.predict(X_test)
false_negatives = X_test[(y_test == 1) & (preds == 0)]
print("False negatives:", len(false_negatives), "of", (y_test==1).sum(), "true declining pages")
print(false_negatives.describe())

ctr_90d                 0.522561
clicks_90d              0.195108
avg_position_90d        0.153550
impressions_90d         0.128781
days_since_last_seen    0.000000
dtype: float64
False negatives: 2093 of 4054 true declining pages
       impressions_90d   clicks_90d  avg_position_90d      ctr_90d  \
count      2093.000000  2093.000000       2093.000000  2093.000000   
mean        594.520306     3.581940         23.831881     0.005305   
std        1235.296511    11.497011         19.210852     0.006314   
min         100.000000     0.000000          0.599376     0.000000   
25%         154.000000     0.000000          7.863496     0.000000   
50%         249.000000     1.000000         17.350642     0.004167   
75%         552.000000     3.000000         36.314221     0.007576   
max       19657.000000   315.000000         77.911546     0.075000   

       days_since_last_seen  
count                2093.0  
mean                    0.0  
std                     0.0  
min               

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.